# M2 Round 3B — VAR Baseline with BIC Lag Selection

**Issue:** #29  
**Owner:** Mitchel  
**Reviewer:** Lerneir  
**Branch:** `artifact/m2-round3b-baseline-var-bic`

## Objective

This notebook implements Round 3 Path B of the baseline-model comparison.

The goal is to evaluate a Vector Autoregression (VAR) model using lag order selected by the Bayesian Information Criterion (BIC), and compare its forecast accuracy against the shared Random Walk (naïve) benchmark.

To preserve a controlled AIC-vs-BIC comparison with Issue #28, the dataset, feature set, transformations, forecast horizons, evaluation procedure, benchmark, and metrics should remain the same. The intended experimental difference is the lag-order selection criterion.

**Fix (per #29 review):** an earlier version of this notebook included `usdcad` in the feature set while `#28`'s script didn't, which confounded the AIC-vs-BIC comparison with an extra input variable. The feature set below now matches `#28`'s exactly, and a Diebold-Mariano significance test (matching `#28`'s, including the `dm_report()` fix from `#43`/`#44`) has been added so both paths are reported at the same rigor.

## Common Round 3 feature set

- `yield_spread_10y_2y`
- `overnight_rate`
- `us_treasury_10y`
- `fed_funds_rate`
- `cpi_yoy`

## Round 2 findings carried into this notebook

The merged Round 2 EDA found that the candidate series are non-stationary 

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.api import VAR

# Resolve project root from notebooks/03_models/
PROJECT_ROOT = Path.cwd().resolve().parents[1]
PROCESSED = PROJECT_ROOT / "data" / "processed"

# Round 3 feature set — matches #28's script exactly (no usdcad; an earlier
# version of this notebook included it, which confounded the AIC-vs-BIC
# comparison with an extra input variable, per #29 review)
FEATURES = [
    "yield_spread_10y_2y",
    "overnight_rate",
    "us_treasury_10y",
    "fed_funds_rate",
    "cpi_yoy",
]

TARGET = "yield_spread_10y_2y"

HORIZONS = [1, 5, 20]

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED)

Project root: /home/nholguin/projects/DAMO-699-Capstone-project-GRP5
Processed data: /home/nholguin/projects/DAMO-699-Capstone-project-GRP5/data/processed


In [2]:
# Load processed datasets
boc = pd.read_csv(
    PROCESSED / "bank_of_canada_data.csv",
    parse_dates=["date"]
)

fred = pd.read_csv(
    PROCESSED / "fred_rates.csv",
    parse_dates=["date"]
)

cpi = pd.read_csv(
    PROCESSED / "statcan_cpi.csv",
    parse_dates=["reference_month", "release_date"]
)

print("BoC:", boc.shape)
print("FRED:", fred.shape)
print("CPI:", cpi.shape)

BoC: (4552, 12)
FRED: (4563, 3)
CPI: (210, 3)


In [3]:
# Merge BoC + FRED on date
daily = (
    boc.merge(fred, on="date", how="outer")
       .sort_values("date")
       .reset_index(drop=True)
)

# Compute CPI YoY at monthly frequency BEFORE expanding to daily
cpi_monthly = cpi.sort_values("reference_month").copy()
cpi_monthly["cpi_yoy"] = (
    cpi_monthly["cpi_all_items"].pct_change(12) * 100
)

# Align CPI by release date to avoid look-ahead bias
cpi_daily = (
    cpi_monthly[
        ["release_date", "cpi_all_items", "cpi_yoy"]
    ]
    .rename(columns={"release_date": "date"})
    .sort_values("date")
)

daily = (
    daily.merge(cpi_daily, on="date", how="left")
         .sort_values("date")
         .reset_index(drop=True)
)

# Once CPI is publicly available, carry the latest released value forward
daily["cpi_all_items"] = daily["cpi_all_items"].ffill()
daily["cpi_yoy"] = daily["cpi_yoy"].ffill()

print("Daily merged shape:", daily.shape)
print("Date range:", daily["date"].min(), "->", daily["date"].max())

daily[
    [
        "date",
        "yield_spread_10y_2y",
        "overnight_rate",
        "usdcad",
        "us_treasury_10y",
        "fed_funds_rate",
        "cpi_yoy",
    ]
].tail()

Daily merged shape: (4563, 16)
Date range: 2009-01-02 00:00:00 -> 2026-06-30 00:00:00


,date,yield_spread_10y_2y,overnight_rate,usdcad,us_treasury_10y,fed_funds_rate,cpi_yoy
4558,2026-06-24,0.63,2.25,1.4234,4.41,3.63,3.225806
4559,2026-06-25,0.64,2.25,1.4204,4.40,3.63,3.225806
4560,2026-06-26,0.64,2.25,1.4186,4.38,3.63,3.225806
4561,2026-06-29,0.64,2.25,1.4206,4.38,3.63,3.225806
4562,2026-06-30,0.64,2.25,1.4210,4.44,3.63,3.225806


In [4]:
# Keep only the agreed Round 3 feature set
model_levels = (
    daily[["date"] + FEATURES]
    .copy()
    .sort_values("date")
    .reset_index(drop=True)
)

# First-difference each series once, following the Round 2 ADF findings
model_diff = model_levels.copy()

for col in FEATURES:
    model_diff[col] = model_diff[col].diff()

# Remove rows that are unavailable after alignment/differencing
model_diff = (
    model_diff
    .dropna(subset=FEATURES)
    .reset_index(drop=True)
)

print("Modeling sample shape:", model_diff.shape)
print("Modeling date range:", model_diff["date"].min(), "->", model_diff["date"].max())

model_diff.head()

Modeling sample shape: (3774, 6)
Modeling date range: 2010-02-19 00:00:00 -> 2026-06-30 00:00:00


,date,yield_spread_10y_2y,overnight_rate,us_treasury_10y,fed_funds_rate,cpi_yoy
0,2010-02-19,-0.01,0.0,-0.01,0.01,0.0
1,2010-02-22,0.01,0.0,0.02,-0.01,0.0
2,2010-02-23,-0.02,0.0,-0.11,0.00,0.0
3,2010-02-24,0.01,0.0,0.01,-0.01,0.0
4,2010-02-25,0.01,0.0,-0.06,0.01,0.0


In [5]:
# Separate dates from stationary model inputs
model_dates = model_diff["date"].copy()

var_data = (
    model_diff[FEATURES]
    .copy()
)

print("VAR input shape:", var_data.shape)
print("Columns:", list(var_data.columns))

VAR input shape: (3774, 5)
Columns: ['yield_spread_10y_2y', 'overnight_rate', 'us_treasury_10y', 'fed_funds_rate', 'cpi_yoy']


In [6]:
# Select VAR lag order using BIC
MAX_LAGS = 15

lag_selection = VAR(var_data).select_order(maxlags=MAX_LAGS)

print(lag_selection.summary())
print("\nSelected lag by BIC:", lag_selection.bic)

 VAR Order Selection (* highlights the minimums)  
       AIC         BIC         FPE         HQIC   
--------------------------------------------------
0       -30.96     -30.95*   3.593e-14      -30.95
1       -30.96      -30.91   3.593e-14      -30.94
2       -30.96      -30.87   3.586e-14      -30.93
3       -30.96      -30.83   3.567e-14      -30.92
4       -30.96      -30.78   3.587e-14      -30.90
5       -30.98      -30.76   3.517e-14      -30.90
6       -30.97      -30.72   3.538e-14      -30.88
7       -30.97      -30.67   3.543e-14      -30.87
8       -30.98      -30.64   3.525e-14      -30.86
9       -30.98      -30.60   3.521e-14      -30.84
10      -31.11      -30.69   3.070e-14     -30.96*
11      -31.11      -30.65   3.074e-14      -30.95
12      -31.11      -30.60   3.096e-14      -30.93
13      -31.11      -30.56   3.087e-14      -30.91
14      -31.10      -30.52   3.102e-14      -30.90
15     -31.12*      -30.49  3.043e-14*      -30.90
-------------------------------

In [7]:
# Shared evaluation configuration
MIN_TRAIN = 500
STEP = 5

BIC_LAG = int(lag_selection.bic)

print("BIC lag:", BIC_LAG)
print("Minimum training observations:", MIN_TRAIN)
print("Evaluation step:", STEP)
print("Horizons:", HORIZONS)

BIC lag: 0
Minimum training observations: 500
Evaluation step: 5
Horizons: [1, 5, 20]


In [8]:
# Align target levels exactly to the final modeling dates
aligned_levels = (
    model_levels[
        model_levels["date"].isin(model_diff["date"])
    ][["date", TARGET]]
    .sort_values("date")
    .reset_index(drop=True)
)

# Safety check: dates must match row-by-row
assert len(aligned_levels) == len(model_diff)
assert aligned_levels["date"].equals(model_diff["date"])

print("Aligned observations:", len(aligned_levels))
print(
    "Aligned date range:",
    aligned_levels["date"].min(),
    "->",
    aligned_levels["date"].max()
)

Aligned observations: 3774
Aligned date range: 2010-02-19 00:00:00 -> 2026-06-30 00:00:00


In [9]:
results = []

target_idx = FEATURES.index(TARGET)

for origin in range(MIN_TRAIN, len(model_diff) - max(HORIZONS), STEP):
    train_diff = model_diff.iloc[:origin][FEATURES].copy()

    # Level corresponding exactly to the forecast origin
    last_level = aligned_levels.loc[origin - 1, TARGET]

    if BIC_LAG == 0:
        # VAR(0): constant expected change estimated from training data
        target_mean_change = train_diff[TARGET].mean()

    else:
        var_model = VAR(train_diff)
        var_fit = var_model.fit(BIC_LAG)

    for h in HORIZONS:
        # h observations ahead from the origin
        actual_level = aligned_levels.loc[origin + h - 1, TARGET]

        # Random Walk
        naive_forecast = last_level

        # VAR-BIC
        if BIC_LAG == 0:
            var_forecast = last_level + h * target_mean_change
        else:
            forecast_diff = var_fit.forecast(
                train_diff.values[-BIC_LAG:],
                steps=h
            )

            cumulative_target_change = forecast_diff[:, target_idx].sum()
            var_forecast = last_level + cumulative_target_change

        results.append({
            "origin_date": aligned_levels.loc[origin - 1, "date"],
            "horizon": h,
            "actual": actual_level,
            "var_bic": var_forecast,
            "naive": naive_forecast,
        })

results_df = pd.DataFrame(results)

print("Forecast rows:", len(results_df))
print(results_df.groupby("horizon").size())

results_df.head()

Forecast rows: 1953
horizon
1     651
5     651
20    651
dtype: int64


,origin_date,horizon,actual,var_bic,naive
0,2012-04-12,1,0.79,0.817720,0.82
1,2012-04-12,5,0.71,0.808600,0.82
2,2012-04-12,20,0.75,0.774400,0.82
3,2012-04-19,1,0.71,0.707525,0.71
4,2012-04-19,5,0.68,0.697624,0.71


In [10]:
metrics = []

for h in HORIZONS:
    subset = results_df[results_df["horizon"] == h]

    var_rmse = np.sqrt(
        mean_squared_error(subset["actual"], subset["var_bic"])
    )
    var_mae = mean_absolute_error(
        subset["actual"], subset["var_bic"]
    )

    naive_rmse = np.sqrt(
        mean_squared_error(subset["actual"], subset["naive"])
    )
    naive_mae = mean_absolute_error(
        subset["actual"], subset["naive"]
    )

    metrics.append({
        "horizon": h,
        "var_bic_rmse": var_rmse,
        "naive_rmse": naive_rmse,
        "var_bic_mae": var_mae,
        "naive_mae": naive_mae,
        "rmse_improvement_pct": (
            (naive_rmse - var_rmse) / naive_rmse * 100
        ),
        "mae_improvement_pct": (
            (naive_mae - var_mae) / naive_mae * 100
        ),
    })

metrics_df = pd.DataFrame(metrics)

metrics_df.round(6)

,horizon,var_bic_rmse,naive_rmse,var_bic_mae,naive_mae,rmse_improvement_pct,mae_improvement_pct
0,1,0.030302,0.030268,0.021908,0.021751,-0.113554,-0.720553
1,5,0.065452,0.065181,0.048325,0.048111,-0.415446,-0.445418
2,20,0.137259,0.134971,0.103700,0.101613,-1.695819,-2.053665


In [11]:
# Diebold-Mariano test — is the RMSE/MAE gap vs. naive statistically significant, or noise?
# Mirrors #28's dm_report() (src/EDA_VAR_AIC _lag _order.py), including the #43/#44 fix:
# report RMSE- and MAE-loss verdicts separately, plus an overall verdict requiring both to agree.
from dieboldmariano import dm_test

ALPHA = 0.05

dm_rows = []
for h in HORIZONS:
    subset = results_df[results_df["horizon"] == h]
    actual = subset["actual"].to_numpy()
    var_pred = subset["var_bic"].to_numpy()
    naive_pred = subset["naive"].to_numpy()

    dm_rmse, p_rmse = dm_test(
        actual, var_pred, naive_pred,
        loss=lambda u, v: (u - v) ** 2,
        h=h, harvey_correction=True, variance_estimator="bartlett",
    )
    dm_mae, p_mae = dm_test(
        actual, var_pred, naive_pred,
        loss=lambda u, v: abs(u - v),
        h=h, harvey_correction=True, variance_estimator="bartlett",
    )

    var_better_rmse = bool(p_rmse < ALPHA and dm_rmse < 0)
    naive_better_rmse = bool(p_rmse < ALPHA and dm_rmse > 0)
    var_better_mae = bool(p_mae < ALPHA and dm_mae < 0)
    naive_better_mae = bool(p_mae < ALPHA and dm_mae > 0)

    dm_rows.append({
        "horizon_days": h,
        "dm_stat_squared_loss": round(dm_rmse, 3),
        "dm_p_value_squared_loss": round(p_rmse, 4),
        "dm_stat_absolute_loss": round(dm_mae, 3),
        "dm_p_value_absolute_loss": round(p_mae, 4),
        "var_significantly_better_rmse": var_better_rmse,
        "naive_significantly_better_rmse": naive_better_rmse,
        "var_significantly_better_mae": var_better_mae,
        "naive_significantly_better_mae": naive_better_mae,
        "var_significantly_better": bool(var_better_rmse and var_better_mae),
        "naive_significantly_better": bool(naive_better_rmse and naive_better_mae),
    })

dm_results_df = pd.DataFrame(dm_rows)

for _, row in dm_results_df.iterrows():
    h = int(row["horizon_days"])
    if row["var_significantly_better"]:
        verdict = "VAR-BIC significantly better than naive (both RMSE and MAE agree)"
    elif row["naive_significantly_better"]:
        verdict = "naive significantly better than VAR-BIC (both RMSE and MAE agree)"
    elif (row["var_significantly_better_rmse"] and row["naive_significantly_better_mae"]) or \
         (row["naive_significantly_better_rmse"] and row["var_significantly_better_mae"]):
        verdict = "mixed result: RMSE and MAE are both significant but identify different winning models"
    elif row["var_significantly_better_rmse"] or row["naive_significantly_better_rmse"]:
        winner = "VAR-BIC" if row["var_significantly_better_rmse"] else "naive"
        verdict = (f"{winner} significantly better on RMSE only "
                   f"(p_rmse={row['dm_p_value_squared_loss']:.4f}) -- MAE test not significant")
    elif row["var_significantly_better_mae"] or row["naive_significantly_better_mae"]:
        winner = "VAR-BIC" if row["var_significantly_better_mae"] else "naive"
        verdict = (f"{winner} significantly better on MAE only "
                   f"(p_mae={row['dm_p_value_absolute_loss']:.4f}) -- RMSE test not significant")
    else:
        verdict = "no significant difference on either RMSE or MAE"
    print(f"  h={h}: {verdict}")

dm_results_df

  h=1: naive significantly better on MAE only (p_mae=0.0003) -- RMSE test not significant
  h=5: no significant difference on either RMSE or MAE
  h=20: no significant difference on either RMSE or MAE


,horizon_days,dm_stat_squared_loss,dm_p_value_squared_loss,dm_stat_absolute_loss,dm_p_value_absolute_loss,var_significantly_better_rmse,naive_significantly_better_rmse,var_significantly_better_mae,naive_significantly_better_mae,var_significantly_better,naive_significantly_better
0,1,0.854,0.3935,3.651,0.0003,False,False,False,True,False,False
1,5,1.287,0.1985,0.954,0.3406,False,False,False,False,False,False
2,20,1.312,0.1899,1.298,0.1947,False,False,False,False,False,False


In [12]:
# Save Round 3B results
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

metrics_output = OUTPUT_DIR / "r3_pathb_var_bic_vs_naive.csv"
forecasts_output = OUTPUT_DIR / "r3_pathb_var_bic_forecasts.csv"
dm_output = OUTPUT_DIR / "r3_pathb_diebold_mariano.csv"

metrics_df.to_csv(metrics_output, index=False)
results_df.to_csv(forecasts_output, index=False)
dm_results_df.to_csv(dm_output, index=False)

print("Saved metrics:", metrics_output)
print("Saved forecasts:", forecasts_output)
print("Saved Diebold-Mariano results:", dm_output)

Saved metrics: /home/nholguin/projects/DAMO-699-Capstone-project-GRP5/outputs/r3_pathb_var_bic_vs_naive.csv
Saved forecasts: /home/nholguin/projects/DAMO-699-Capstone-project-GRP5/outputs/r3_pathb_var_bic_forecasts.csv
Saved Diebold-Mariano results: /home/nholguin/projects/DAMO-699-Capstone-project-GRP5/outputs/r3_pathb_diebold_mariano.csv


## Round 3B Conclusion — VAR with BIC Lag Selection

Using the Round 3 feature set (aligned with `#28`: `yield_spread_10y_2y`, `overnight_rate`, `us_treasury_10y`, `fed_funds_rate`, `cpi_yoy` — an earlier version of this notebook also included `usdcad`, which confounded the AIC-vs-BIC comparison; removed per `#29` review), all five variables were modeled in first differences based on the Round 2 stationarity findings. The final modeling sample contained 3,774 aligned daily observations.

For the BIC-based VAR specification, lag-order selection over a maximum of 15 lags selected:

- **BIC lag order: 0**

This indicates that, under the BIC penalty, adding autoregressive lags did not provide enough additional explanatory value to justify the increase in model complexity. At lag 0 the specification reduces to a constant expected daily change estimated from the training window — it does not use the other variables' lagged values at all, which is also why dropping `usdcad` from the feature set left the RMSE/MAE numbers below essentially unchanged from the earlier draft.

The BIC-selected specification was evaluated using an expanding-window approach with forecasts generated every 5 observations after an initial training sample of 500 observations. Performance was assessed at **1-day, 5-day, and 20-day horizons** using RMSE and MAE, with a Random Walk forecast used as the shared naïve benchmark.

### Forecast performance

| Horizon | VAR-BIC RMSE | Random Walk RMSE | VAR-BIC MAE | Random Walk MAE |
|---|---:|---:|---:|---:|
| 1 day | 0.030302 | 0.030268 | 0.021908 | 0.021751 |
| 5 days | 0.065452 | 0.065181 | 0.048325 | 0.048111 |
| 20 days | 0.137259 | 0.134971 | 0.103700 | 0.101613 |

The BIC specification did **not outperform the Random Walk benchmark at any forecast horizon**. Relative to the naïve benchmark, the VAR-BIC model produced slightly higher errors:

- **1-day horizon:** RMSE +0.11%, MAE +0.72%
- **5-day horizon:** RMSE +0.42%, MAE +0.45%
- **20-day horizon:** RMSE +1.70%, MAE +2.05%

### Diebold-Mariano significance (added per `#29` review, matching `#28`'s methodology)

Point RMSE/MAE differences above don't say whether the gap from naive is real or sampling noise. Running the same Diebold-Mariano test `#28` uses (Bartlett-weighted long-run variance, Harvey small-sample correction), separately per loss function:

| Horizon | DM stat (RMSE) | p (RMSE) | DM stat (MAE) | p (MAE) | Verdict |
|---|---:|---:|---:|---:|---|
| 1 day | 0.854 | 0.3935 | 3.651 | 0.0003 | naive significantly better on MAE only |
| 5 days | 1.287 | 0.1985 | 0.954 | 0.3406 | no significant difference |
| 20 days | 1.312 | 0.1899 | 1.298 | 0.1947 | no significant difference |

At h=1, naive is significantly more accurate on MAE (p=0.0003) — closely replicating `#28`'s own AIC-path DM finding (naive significantly better on MAE at h=1, p=0.0005). Neither path finds VAR significantly better than naive at any horizon, on either loss function.

Overall, BIC strongly favored model parsimony and selected a zero-lag specification. The resulting forecasts were very close to the Random Walk benchmark, and the DM test confirms the naive benchmark is not just numerically ahead but *statistically* ahead at the shortest horizon. These results provide the Path B baseline for the team comparison between **AIC- and BIC-based VAR lag selection** — see `M2_CHECKLIST.md` for the recorded Round 3 convergence decision (keeping both as a documented sensitivity check, since both agree VAR does not beat naive here).